# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import pandas as pd  # noqa: F401
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402, F401
from src.database import SessionLocal, engine  # noqa: E402, F401
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

In [ ]:
import src.database.models as models
import src.portfolio.purchase as purchase
reload(purchase)

paths = {
    "personas": "../data/PERSONAS.CSV",
    "prestamos": "../data/PRESTAMOS.CSV",
    "cuotas": "../data/CUOTAS.CSV"}

NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mentiritas S.A.", 30713257880)
NewRelation = models.Relacion()
NewRelation.add_single_mapping(1, "provincias", 1, 2)
NewRelation.add_single_mapping(1, "provincias", 3, 6)
NewRelation.add_single_mapping(1, "provincias", 7, 5)
NewRelation.add_single_mapping(1, "provincias", 16, 16)
NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mutual Uno", 30713257870)
NewAssociate.create_socio("Mutual Dos", 30713257860)
NewAssociate.create_socio("Mutual Tres", 30713157860)
NewRelation.add_single_mapping(1, "socios_comerciales", 2, 14)
NewRelation.add_single_mapping(1, "socios_comerciales", 3, 20)
NewRelation.add_single_mapping(1, "socios_comerciales", 4, 7)
NewFolder_1 = purchase.PortfolioPurchase()
NewFolder_1.process_full_portfolio("Folder Test 1", "2026/05/31", 0.45, 30713257880, "Mentiritas S.A.", paths=paths, recurso=False, iva=True)

paths = {'personas': 'D:/OneDrive - Estudio Scoccia/FIDEISA 1 - FINANCIERA/CREDISE S.A/CESION DE CARTERA A FCI/Venta Nro. 008 - CFL 0271/271_PERSONAS.CSV', 'prestamos': 'D:/OneDrive - Estudio Scoccia/FIDEISA 1 - FINANCIERA/CREDISE S.A/CESION DE CARTERA A FCI/Venta Nro. 008 - CFL 0271/271_PRESTAMOS.CSV', 'cuotas': 'D:/OneDrive - Estudio Scoccia/FIDEISA 1 - FINANCIERA/CREDISE S.A/CESION DE CARTERA A FCI/Venta Nro. 008 - CFL 0271/271_CUOTAS.CSV'}

NewFolder_2 = purchase.PortfolioPurchase()
NewFolder_2.process_full_portfolio("Folder Test 2", "2026/06/03", 0.44, 30713257880, "Mentiritas S.A.", paths=paths, recurso=True, iva=False);

In [ ]:
import src.reports.balances as calcular
reload(calcular)

df = calcular.saldos("2026/05/31", con_saldo=False, propias=True, agrupar=False, socios=True, vencimientos=True, recurso=True, iva=True)
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df.sort_values(by="Fecha Vencimiento")

In [ ]:
import src.logic.collections as collections
reload(collections)

NewColl = collections.CollectionManager()
df = NewColl.process_massive_collection("CLIENTE_CUIL", "F", "N", payment_date="2026/06/05", path='D:/Repositorios/Credit_Manager/data/Cobranza - Mutual Uno - 2026-05-28.xlsx', early=False)

df

In [ ]:
import src.reports.balances as calcular
reload(calcular)

df = calcular.saldos("2026/06/05", con_saldo=True, propias=True, agrupar=False, socios=True, vencimientos=True, recurso=True, iva=True)
# df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df.sort_values(by=["Fecha Vencimiento"])

In [ ]:
import src.logic.collections as collections
reload(collections)

NewColl = collections.CollectionManager()
df = NewColl.process_massive_collection("CLIENTE_CUIL", "F", "N", payment_date="2026/07/05", path='D:/Repositorios/Credit_Manager/data/Cobranza - Mutual Uno - 2026-06-28.xlsx', early=True)

df

In [ ]:
import src.reports.balances as calcular
reload(calcular)

df = calcular.saldos("2026/09/05", con_saldo=True, propias=True, agrupar=False, socios=True, vencimientos=True, recurso=True, iva=True)
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df.loc[(df["Originador"] == "Mutual Uno") & (df["ID Cartera"] == 1)].sort_values(by=["Fecha Vencimiento"])